In [1]:
import os
import pandas as pd
import numpy as np
import xarray as xr
import dask
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

# ============================
# User settings
# ============================
sdate, edate = '20090701', '20240630'
write_path = '/scratch/ng72/ms5578/solar_wind_tseries/'

# BARRA-R2 variable paths
u_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/ua100m/latest/"
v_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/va100m/latest/"

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")
cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv")

# Dask cluster

client = Client()
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.08/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36303 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/36303/status,
Dashboard: /proxy/36303/status,Workers: 7
Total threads: 7,Total memory: 32.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41723,Workers: 0
Dashboard: /proxy/36303/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:45113,Total threads: 1
Dashboard: /proxy/37333/status,Memory: 4.57 GiB
Nanny: tcp://127.0.0.1:40481,


In [2]:
boco = cluster_dates[cluster_dates['BOCORWF1'] == 1]
non = cluster_dates[cluster_dates['BOCORWF1'] == 0]

In [3]:
def get_days(days, nc_dir):
    date_list = pd.to_datetime(days['date'])

    # Build filename filter
    all_files = os.listdir(nc_dir)
    selected_files = [
        os.path.join(nc_dir, f)
        for f in all_files
        if any(d.strftime("%Y%m") in f for d in date_list)
    ]

    # Open multiple files lazily with parallel reads
    ds = xr.open_mfdataset(
        selected_files,
        combine='by_coords',
        parallel=True,
        chunks='auto'
    )

    # Select all hours of the requested dates
    subset = ds.where(ds.time.dt.floor('D').isin(date_list), drop=True)

    return subset


In [4]:
u_clust = get_days(non, u_path)
v_clust = get_days(non, v_path)
u_boco = get_days(boco, u_path)
v_boco = get_days(boco, v_path)

boco = xr.merge([u_boco, v_boco])
clust = xr.merge([u_clust, v_clust])

In [5]:
# Lazy write — computation triggered only once
with ProgressBar():
    dask.compute(
        boco.to_netcdf("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/boco_wind_vec.nc", compute=False),
        clust.to_netcdf("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/cluster_wind_vec.nc", compute=False)
    )

HDF5-DIAG: Error detected in HDF5 (1.14.3) thread 0:
  #000: H5F.c line 660 in H5Fcreate(): unable to synchronously create file
    major: File accessibility
    minor: Unable to create file
  #001: H5F.c line 614 in H5F__create_api_common(): unable to create file
    major: File accessibility
    minor: Unable to open file
  #002: H5VLcallback.c line 3605 in H5VL_file_create(): file create failed
    major: Virtual Object Layer
    minor: Unable to create file
  #003: H5VLcallback.c line 3571 in H5VL__file_create(): file create failed
    major: Virtual Object Layer
    minor: Unable to create file
  #004: H5VLnative_file.c line 94 in H5VL__native_file_create(): unable to create file
    major: File accessibility
    minor: Unable to open file
  #005: H5Fint.c line 1910 in H5F_open(): unable to lock the file
    major: File accessibility
    minor: Unable to lock file
  #006: H5FD.c line 2412 in H5FD_lock(): driver lock request failed
    major: Virtual File Layer
    minor: Unable to

PermissionError: [Errno 13] Permission denied: '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/boco_wind_vec.nc'